# Data Acquisition

In this lab, we will explore the process of data acquisition. 

In the case of *passive* network traffic analysis, there are generally two primary ways of acquiring data:
* Packet capture
* Network traffic flows (sometimes called IPFIX)

The advent of more programmability in networks is quickly changing this landscape. 

In particular, systems like Retina are now making it possible to ask more complex questions of network traffic from passive traffic capture and analysis but the general underlying traffic patterns are still based on raw packet capture.

## Background

Because packet captures are so large, it can sometimes be convenient to work with summary statistics about network traffic. Instead of the raw packets, data could represent the total number of bytes, packets, and so forth for flows. 

Raw traffic capture is thus sometimes represented as summaries of flow statistics, rather than raw packet traces. In this activity, we will *generate* the summary statistics and then think about what types of information is (and is not) available in a packet trace summary vs. a raw packet capture.

## Step 1: Load a Packet Trace

Load the packet capture from the last assignment.

In [1]:
import pandas as pd

ndf = pd.read_csv("data/netflix.csv.gz")
ndf.head(20)

,No.,Time,Source,Destination,Protocol,Length,Info
0,1,2018-02-11 08:10:00.534682,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0xed0c A fonts.gstatic.com
1,2,2018-02-11 08:10:00.534832,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0x301a AAAA fonts.gstatic.com
2,3,2018-02-11 08:10:00.539408,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x11d3 A googleads.g.doubleclic...
3,4,2018-02-11 08:10:00.541204,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x1284 AAAA googleads.g.doublec...
4,5,2018-02-11 08:10:00.545785,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,78,Standard query 0x3432 AAAA ytimg.l.google.com
5,6,2018-02-11 08:10:00.547036,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,96,Standard query 0xb756 A r4---sn-gxo5uxg-jqbe.g...
6,7,2018-02-11 08:10:00.547156,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0x62ab A ssl.gstatic.com
7,8,2018-02-11 08:10:00.547249,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,74,Standard query 0x42fb A www.google.com
8,9,2018-02-11 08:10:00.853950,ns-vip-pro.paris.inria.fr,192.168.43.72,DNS,386,Standard query response 0x11d3 A 216.58.213.162
9,10,2018-02-11 08:10:00.853970,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0x8756 A www.gstatic.com


## Step 2: Generate Statistics for Each Flow

A **flow** is defined as groups of packets that share the following attributes:
* Source IP Address
* Destination IP Address
* Source Port
* Destination Port
* Time interval

The csv file we used in the past assignments do not have port numbers, so you can simply group on source and destination IP address.

For each flow in the packet trace, generate the following statistics for each flow:

* Number of bytes
* Number of packets
* Duration (time)

In [2]:
df = ndf[['Source', 'Destination', 'Length', 'Time']].dropna()

# Convert Length to numeric (in case it's read as string)
df['Length'] = pd.to_numeric(df['Length'], errors='coerce')

# Convert Time to datetime or numeric
df['Time'] = pd.to_datetime(df['Time'], errors='coerce')

# === Group by Source and Destination ===
flows = df.groupby(['Source', 'Destination']).agg(
    num_packets=('Length', 'count'),
    total_bytes=('Length', 'sum'),
    start_time=('Time', 'min'),
    end_time=('Time', 'max')
).reset_index()

# === Compute duration in seconds ===
flows['duration_sec'] = (flows['end_time'] - flows['start_time']).dt.total_seconds().clip(lower=0)

### Total Number of Flows

Count the total number of flows in this trace.

In [3]:
total_flows = len(flows)
print(f"Total number of flows: {total_flows}")

Total number of flows: 77


In [5]:
flows.head(20)

,Source,Destination,num_packets,total_bytes,start_time,end_time,duration_sec
0,0.0.0.0,255.255.255.255,8,3032,2018-02-11 08:10:37.716265,2018-02-11 08:18:04.797274,447.081009
1,0.0.0.0,all-systems.mcast.net,4,184,2018-02-11 08:10:52.052108,2018-02-11 08:17:08.306861,376.254753
2,104.31.113.215,192.168.43.72,6,934,2018-02-11 08:18:13.613768,2018-02-11 08:18:16.299097,2.685329
3,17.188.166.20,192.168.43.72,2,218,2018-02-11 08:14:10.039478,2018-02-11 08:14:17.674745,7.635267
4,17.252.44.15,192.168.43.72,3,420,2018-02-11 08:11:53.050694,2018-02-11 08:16:10.425198,257.374504
5,192.168.1.159,192.168.43.72,8,752,2018-02-11 08:12:31.587688,2018-02-11 08:12:31.588515,0.000827
6,192.168.43.192,224.0.0.251,31,7072,2018-02-11 08:10:36.696229,2018-02-11 08:17:58.041732,441.345503
7,192.168.43.37,224.0.0.251,10,882,2018-02-11 08:11:40.385262,2018-02-11 08:18:09.917256,389.531994
8,192.168.43.72,104.31.113.215,8,814,2018-02-11 08:18:13.600579,2018-02-11 08:18:16.299136,2.698557
9,192.168.43.72,17.188.166.20,3,442,2018-02-11 08:14:03.692513,2018-02-11 08:14:17.675034,13.982521


### Number of Bytes

Count the total number of bytes for each flow in the trace. 

Then, sort the flows by size, in bytes.  

What do you notice about the large flows? What do they look like?

In [7]:
flows_sorted = flows.sort_values('total_bytes', ascending=False)
flows_sorted.head(20)

,Source,Destination,num_packets,total_bytes,start_time,end_time,duration_sec
67,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,80084,120607242,2018-02-11 08:10:20.958591,2018-02-11 08:18:16.298969,475.340378
66,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,4873,7138148,2018-02-11 08:10:20.957155,2018-02-11 08:18:16.299010,475.341855
25,192.168.43.72,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,47902,3357228,2018-02-11 08:10:20.820787,2018-02-11 08:18:16.299106,475.478319
44,a23-57-80-120.deploy.static.akamaitechnologies...,192.168.43.72,1005,1332086,2018-02-11 08:10:03.629125,2018-02-11 08:16:16.525640,372.896515
65,ipv4-c063-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,338,431178,2018-02-11 08:11:01.246971,2018-02-11 08:12:16.340228,75.093257
46,ec2-52-19-39-146.eu-west-1.compute.amazonaws.com,192.168.43.72,472,348141,2018-02-11 08:10:03.671653,2018-02-11 08:17:13.743144,430.071491
19,192.168.43.72,ec2-52-19-39-146.eu-west-1.compute.amazonaws.com,489,340330,2018-02-11 08:10:02.903625,2018-02-11 08:17:13.743191,430.839566
24,192.168.43.72,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,3170,244455,2018-02-11 08:10:20.811333,2018-02-11 08:18:16.299106,475.487773
37,198.38.120.137,192.168.43.72,101,116255,2018-02-11 08:10:05.890178,2018-02-11 08:11:13.213402,67.323224
75,par10s38-in-f3.1e100.net,192.168.43.72,144,98271,2018-02-11 08:10:01.209115,2018-02-11 08:18:16.313540,495.104425


There’s a clear asymmetry between directions: the largest flows are from Netflix’s content delivery network (CDN) → local machine, not the other way around.

What this tells us:
A few large flows dominate total traffic. Most bytes come from just a handful of flows from Netflix CDN servers.
Download-heavy behavior. This is expected for video streaming, where the client receives much more data than it sends.
Long-lived flows. These flows last several minutes (475 seconds ≈ 8 minutes), typical of continuous media streaming.
Many smaller flows coexist. Other flows (Amazon, Akamai, etc.) are much smaller in total bytes — possibly control, ads, or background services.
This reflects a heavy-tailed distribution, common in real network traces:

### Number of Packets

Count the number of packets in each flow. 

What do you notice about these flows? Are they similar to the largest flows by bytes? Which differ?

In [12]:
flows_by_packets = flows.sort_values('num_packets', ascending=False)
flows_by_packets.head(20)


,Source,Destination,num_packets,total_bytes,start_time,end_time,duration_sec
67,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,80084,120607242,2018-02-11 08:10:20.958591,2018-02-11 08:18:16.298969,475.340378
25,192.168.43.72,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,47902,3357228,2018-02-11 08:10:20.820787,2018-02-11 08:18:16.299106,475.478319
66,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,4873,7138148,2018-02-11 08:10:20.957155,2018-02-11 08:18:16.299010,475.341855
24,192.168.43.72,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,3170,244455,2018-02-11 08:10:20.811333,2018-02-11 08:18:16.299106,475.487773
44,a23-57-80-120.deploy.static.akamaitechnologies...,192.168.43.72,1005,1332086,2018-02-11 08:10:03.629125,2018-02-11 08:16:16.525640,372.896515
17,192.168.43.72,a23-57-80-120.deploy.static.akamaitechnologies...,834,60463,2018-02-11 08:10:02.905014,2018-02-11 08:16:16.525757,373.620743
19,192.168.43.72,ec2-52-19-39-146.eu-west-1.compute.amazonaws.com,489,340330,2018-02-11 08:10:02.903625,2018-02-11 08:17:13.743191,430.839566
46,ec2-52-19-39-146.eu-west-1.compute.amazonaws.com,192.168.43.72,472,348141,2018-02-11 08:10:03.671653,2018-02-11 08:17:13.743144,430.071491
65,ipv4-c063-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,338,431178,2018-02-11 08:11:01.246971,2018-02-11 08:12:16.340228,75.093257
23,192.168.43.72,ipv4-c063-cdg001-ix.1.oca.nflxvideo.net,266,23161,2018-02-11 08:11:01.204076,2018-02-11 08:12:11.774059,70.569983


streaming flows — they’re large in both bytes and packets because they sustain a continuous data transfer.

### Duration

Compute the duration of each flow, by taking the time of the last packet and subtracting the time of the first, for each flow.  

What are the longest flows in the trace?

In [11]:
flows['duration_sec'] = (flows['end_time'] - flows['start_time']).dt.total_seconds().clip(lower=0)
flows_by_duration = flows.sort_values('duration_sec', ascending=False)
flows_by_duration.head(10)

,Source,Destination,num_packets,total_bytes,start_time,end_time,duration_sec
33,192.168.43.72,par10s38-in-f3.1e100.net,158,17828,2018-02-11 08:10:00.861944,2018-02-11 08:18:16.313632,495.451688
75,par10s38-in-f3.1e100.net,192.168.43.72,144,98271,2018-02-11 08:10:01.209115,2018-02-11 08:18:16.313540,495.104425
70,ns-vip-pro.paris.inria.fr,192.168.43.72,50,20208,2018-02-11 08:10:00.853950,2018-02-11 08:18:13.599825,492.745875
28,192.168.43.72,ns-vip-pro.paris.inria.fr,58,4711,2018-02-11 08:10:00.534682,2018-02-11 08:18:13.222075,492.687393
35,192.168.43.97,224.0.0.251,9,1004,2018-02-11 08:10:03.514149,2018-02-11 08:18:10.219615,486.705466
16,192.168.43.72,224.0.0.251,16,2528,2018-02-11 08:10:03.808095,2018-02-11 08:18:10.208890,486.400795
64,fe80::e6ce:8fff:fe01:4c54,ff02::fb,16,2928,2018-02-11 08:10:03.808330,2018-02-11 08:18:10.209063,486.400733
18,192.168.43.72,ec2-34-252-77-54.eu-west-1.compute.amazonaws.com,25,3753,2018-02-11 08:10:12.323784,2018-02-11 08:18:12.876528,480.552744
45,ec2-34-252-77-54.eu-west-1.compute.amazonaws.com,192.168.43.72,21,4884,2018-02-11 08:10:12.468557,2018-02-11 08:18:12.876416,480.407859
24,192.168.43.72,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,3170,244455,2018-02-11 08:10:20.811333,2018-02-11 08:18:16.299106,475.487773


## Bytes and Packets Per Second

Compute the bytes per second and packets per second for each flow.

For a simple feature computation, compute the average bytes and packets per second for each flow, for the entire duration of the flow.  If you want to get more clever or fancy, you can do "windowed averages", computing bytes or packets per second for shorter time intervals.

In [16]:
flows['duration_sec'] = flows['duration_sec'].replace(0, pd.NA)

# --- Compute averages ---
flows['bytes_per_sec'] = flows['total_bytes'] / flows['duration_sec']
flows['packets_per_sec'] = flows['num_packets'] / flows['duration_sec']

# Sort by highest bytes per second
flows_sorted_bps = flows.sort_values('bytes_per_sec', ascending=False)


# Sort by highest packets per second
flows_sorted_pps = flows.sort_values('packets_per_sec', ascending=False)
flows_sorted_bps.head(10)

,Source,Destination,num_packets,total_bytes,start_time,end_time,duration_sec,bytes_per_sec,packets_per_sec
5,192.168.1.159,192.168.43.72,8,752,2018-02-11 08:12:31.587688,2018-02-11 08:12:31.588515,0.000827,909310.76179,9673.518742
67,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,80084,120607242,2018-02-11 08:10:20.958591,2018-02-11 08:18:16.298969,475.340378,253728.165294,168.477166
66,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,4873,7138148,2018-02-11 08:10:20.957155,2018-02-11 08:18:16.299010,475.341855,15016.872436,10.251569
25,192.168.43.72,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,47902,3357228,2018-02-11 08:10:20.820787,2018-02-11 08:18:16.299106,475.478319,7060.738347,100.744867
65,ipv4-c063-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,338,431178,2018-02-11 08:11:01.246971,2018-02-11 08:12:16.340228,75.093257,5741.900368,4.50107
44,a23-57-80-120.deploy.static.akamaitechnologies...,192.168.43.72,1005,1332086,2018-02-11 08:10:03.629125,2018-02-11 08:16:16.525640,372.896515,3572.267228,2.695118
38,198.38.120.153,192.168.43.72,63,39507,2018-02-11 08:10:03.620252,2018-02-11 08:10:17.371280,13.751028,2873.021566,4.581476
76,par21s03-in-f2.1e100.net,192.168.43.72,15,6897,2018-02-11 08:18:13.612362,2018-02-11 08:18:16.298691,2.686329,2567.444271,5.583828
37,198.38.120.137,192.168.43.72,101,116255,2018-02-11 08:10:05.890178,2018-02-11 08:11:13.213402,67.323224,1726.818668,1.500225
69,ipv4-c197-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,69,90447,2018-02-11 08:10:20.949003,2018-02-11 08:11:21.900870,60.951867,1483.908606,1.132041


In [17]:
flows_sorted_pps.head(10)

,Source,Destination,num_packets,total_bytes,start_time,end_time,duration_sec,bytes_per_sec,packets_per_sec
5,192.168.1.159,192.168.43.72,8,752,2018-02-11 08:12:31.587688,2018-02-11 08:12:31.588515,0.000827,909310.76179,9673.518742
67,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,80084,120607242,2018-02-11 08:10:20.958591,2018-02-11 08:18:16.298969,475.340378,253728.165294,168.477166
25,192.168.43.72,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,47902,3357228,2018-02-11 08:10:20.820787,2018-02-11 08:18:16.299106,475.478319,7060.738347,100.744867
66,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,4873,7138148,2018-02-11 08:10:20.957155,2018-02-11 08:18:16.299010,475.341855,15016.872436,10.251569
34,192.168.43.72,par21s03-in-f2.1e100.net,20,2338,2018-02-11 08:18:13.587377,2018-02-11 08:18:16.298784,2.711407,862.28294,7.376244
24,192.168.43.72,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,3170,244455,2018-02-11 08:10:20.811333,2018-02-11 08:18:16.299106,475.487773,514.114166,6.666838
76,par21s03-in-f2.1e100.net,192.168.43.72,15,6897,2018-02-11 08:18:13.612362,2018-02-11 08:18:16.298691,2.686329,2567.444271,5.583828
14,192.168.43.72,198.38.120.153,75,7140,2018-02-11 08:10:02.904561,2018-02-11 08:10:17.371374,14.466813,493.543395,5.184279
38,198.38.120.153,192.168.43.72,63,39507,2018-02-11 08:10:03.620252,2018-02-11 08:10:17.371280,13.751028,2873.021566,4.581476
65,ipv4-c063-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,338,431178,2018-02-11 08:11:01.246971,2018-02-11 08:12:16.340228,75.093257,5741.900368,4.50107


## Note

Some of the libraries that we will use in this class, including the `netml` library from the University of Chicago, will compute these and other statistics automatically.

## Thought Questions

1. What are the largest flows in terms of: Number of bytes? Number of packets?

2. What do you notice about the flow sizes and the directions of flows?

3. What kinds of features are *not* available in packet summary statistics like those above which might be available in a raw packet trace? How might those features be useful for different packet classification problems?

By bytes:
The largest flows are from


"ipv4-c071-cdg001-ix.1.oca.nflxvideo.net → 192.168.43.72
ipv4-c069-cdg001-ix.1.oca.nflxvideo.net → 192.168.43.72"



These Netflix CDN flows transferred tens to hundreds of MB over ~8 minutes.
✅ These are the main downstream video streaming flows.

By packets:
These flows are also at the top of the packet count ranking.
But importantly,
192.168.43.72 → ipv4-c071-cdg001-ix.1.oca.nflxvideo.net
192.168.43.72 → ipv4-c069-cdg001-ix.1.oca.nflxvideo.net
also appear very high because of many small ACK/control packets sent back to the server.